# Kriging Interpolation

The `kriging()` function performs Ordinary Kriging, a geostatistical interpolation method that produces optimal, unbiased predictions from scattered point observations. Unlike IDW, kriging accounts for the spatial correlation structure of the data through a variogram model.

Key features:
- Automatic experimental variogram computation and model fitting
- Three variogram models: spherical, exponential, gaussian
- Optional kriging variance (prediction uncertainty) output
- All four backends: NumPy, Dask, CuPy, Dask+CuPy

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.interpolate import kriging

## 1. Basic interpolation from point observations

Generate scattered sample points from a known surface and use kriging to reconstruct the full field.

In [ ]:
# True surface: z = sin(x) * cos(y)
rng = np.random.RandomState(42)
n_pts = 40
x_pts = rng.uniform(0, 6, n_pts)
y_pts = rng.uniform(0, 6, n_pts)
z_pts = np.sin(x_pts) * np.cos(y_pts) + rng.normal(0, 0.05, n_pts)

# Output grid
x_grid = np.linspace(0, 6, 60)
y_grid = np.linspace(0, 6, 60)
template = xr.DataArray(
    np.zeros((len(y_grid), len(x_grid))),
    dims=['y', 'x'],
    coords={'y': y_grid, 'x': x_grid},
)

result = kriging(x_pts, y_pts, z_pts, template)

# True surface for comparison
gx, gy = np.meshgrid(x_grid, y_grid)
true_surface = np.sin(gx) * np.cos(gy)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(true_surface, extent=[0, 6, 6, 0], cmap='viridis')
axes[0].scatter(x_pts, y_pts, c='red', s=15, edgecolors='k', linewidth=0.5)
axes[0].set_title('True surface + sample points')
fig.colorbar(im0, ax=axes[0], shrink=0.7)

im1 = axes[1].imshow(result.values, extent=[0, 6, 6, 0], cmap='viridis')
axes[1].set_title('Kriging prediction')
fig.colorbar(im1, ax=axes[1], shrink=0.7)

plt.tight_layout()
plt.show()

## 2. Kriging variance

Set `return_variance=True` to get prediction uncertainty. Variance is low near observed points and higher in data-sparse regions.

In [ ]:
pred, var = kriging(x_pts, y_pts, z_pts, template, return_variance=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(pred.values, extent=[0, 6, 6, 0], cmap='viridis')
axes[0].scatter(x_pts, y_pts, c='red', s=15, edgecolors='k', linewidth=0.5)
axes[0].set_title('Prediction')
fig.colorbar(im0, ax=axes[0], shrink=0.7)

im1 = axes[1].imshow(var.values, extent=[0, 6, 6, 0], cmap='magma')
axes[1].scatter(x_pts, y_pts, c='cyan', s=15, edgecolors='k', linewidth=0.5)
axes[1].set_title('Kriging variance')
fig.colorbar(im1, ax=axes[1], shrink=0.7)

plt.tight_layout()
plt.show()

## 3. Variogram model comparison

The `variogram_model` parameter controls the spatial correlation model. Different models produce subtly different predictions.

In [ ]:
models = ['spherical', 'exponential', 'gaussian']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, model in zip(axes, models):
    r = kriging(x_pts, y_pts, z_pts, template, variogram_model=model)
    im = ax.imshow(r.values, extent=[0, 6, 6, 0], cmap='viridis')
    ax.set_title(f'{model}')
    fig.colorbar(im, ax=ax, shrink=0.7)

plt.suptitle('Variogram model comparison', y=1.02)
plt.tight_layout()
plt.show()

## 4. Practical example: soil property mapping

Simulate soil pH measurements at random field locations and produce a continuous map with uncertainty.

In [ ]:
# Simulated soil pH: smooth trend + spatially correlated noise
rng = np.random.RandomState(7)
n_samples = 50
x_soil = rng.uniform(0, 100, n_samples)  # meters
y_soil = rng.uniform(0, 100, n_samples)

# Trend: pH increases toward the northeast
ph = 5.5 + 0.015 * x_soil + 0.010 * y_soil + rng.normal(0, 0.3, n_samples)

# Dense prediction grid
xg = np.linspace(0, 100, 80)
yg = np.linspace(0, 100, 80)
template_soil = xr.DataArray(
    np.zeros((len(yg), len(xg))),
    dims=['y', 'x'],
    coords={'y': yg, 'x': xg},
)

ph_pred, ph_var = kriging(
    x_soil, y_soil, ph, template_soil,
    variogram_model='spherical', return_variance=True,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(
    ph_pred.values, extent=[0, 100, 100, 0],
    cmap='RdYlGn', vmin=5, vmax=8,
)
axes[0].scatter(x_soil, y_soil, c=ph, cmap='RdYlGn', vmin=5, vmax=8,
                s=30, edgecolors='k', linewidth=0.5)
axes[0].set_title('Predicted soil pH')
axes[0].set_xlabel('East (m)')
axes[0].set_ylabel('North (m)')
fig.colorbar(im0, ax=axes[0], shrink=0.7, label='pH')

im1 = axes[1].imshow(
    ph_var.values, extent=[0, 100, 100, 0], cmap='magma',
)
axes[1].scatter(x_soil, y_soil, c='cyan', s=15, edgecolors='k', linewidth=0.5)
axes[1].set_title('Prediction variance')
axes[1].set_xlabel('East (m)')
fig.colorbar(im1, ax=axes[1], shrink=0.7, label='Variance')

plt.tight_layout()
plt.show()